<a href="https://colab.research.google.com/github/hanahmdd/deep-learning-adult-income/blob/main/Deep_Learning_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Deep Learning Practical Assignment (Adult Income Dataset)

## 📌 Dataset
We will use the **Adult Income dataset** (also known as the Census Income dataset).  
The task is to predict whether a person earns **more than $50K/year** based on demographic and employment attributes.

---


In [1]:

from sklearn.datasets import fetch_openml

import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load dataset from OpenML
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame

print(df.head())
print(df.shape)  # (48842, 15)




   age  workclass  fnlwgt     education  education-num      marital-status  \
0   25    Private  226802          11th              7       Never-married   
1   38    Private   89814       HS-grad              9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm             12  Married-civ-spouse   
3   44    Private  160323  Some-college             10  Married-civ-spouse   
4   18        NaN  103497  Some-college             10       Never-married   

          occupation relationship   race     sex  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  class  
0       

In [2]:

X = df.drop(columns="class")
y = df["class"]

In [3]:

y = (y == ">50K").astype(int)

In [4]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

## Part 0: Data Preparation
1. Load the dataset into a DataFrame.
2. Split the data into **training, validation, and test sets**.  
   - Suggested: 70% training, 15% validation, 15% test.
3. Apply any necessary preprocessing:
   - Handle categorical features (encoding).
   - Scale numerical features if needed.
4. After training your models, always report results on:
   - **Training accuracy**
   - **Validation accuracy**
   - **Test accuracy**
5. At the end of the assignment, **compare all methods** across train, validation, and test sets.


In [5]:
numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [6]:

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

In [7]:
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

In [8]:
def build_model(input_dim):

    model = keras.Sequential([
        layers.Dense(64, activation="relu", input_shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    return model


## Part 1: Optimizers
1. Train the same neural network using:
   - Stochastic Gradient Descent (SGD)
   - SGD with Momentum
   - Adam
2. Compare the training and validation accuracy for each optimizer.
3. Which optimizer converges the fastest? Which gives the best generalization?
4. Explain *why* Adam often performs better than plain SGD.

---


In [9]:
optimizers = {
    "SGD": keras.optimizers.SGD(learning_rate=0.01),
    "SGD_Momentum": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "Adam": keras.optimizers.Adam(learning_rate=0.001)
}

results = {}

for name, opt in optimizers.items():

    model = build_model(X_train.shape[1])

    model.compile(
        optimizer=opt,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=32,
        verbose=1
    )

    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]

    results[name] = (train_acc, val_acc, test_acc)

print(results)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.7524 - loss: 0.5076 - val_accuracy: 0.8417 - val_loss: 0.3493
Epoch 2/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8377 - loss: 0.3445 - val_accuracy: 0.8522 - val_loss: 0.3263
Epoch 3/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8410 - loss: 0.3359 - val_accuracy: 0.8580 - val_loss: 0.3168
Epoch 4/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8496 - loss: 0.3214 - val_accuracy: 0.8580 - val_loss: 0.3130
Epoch 5/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8536 - loss: 0.3144 - val_accuracy: 0.8586 - val_loss: 0.3108
Epoch 6/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8525 - loss: 0.3141 - val_accuracy: 0.8591 - val_loss: 0.3099
Epoch 7/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8519 - loss: 0.3162 - val_accuracy: 0.8601 - val_loss: 0.3089
Epoch 8/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8555 - loss: 0.3075 - 

## Part 2: Batch Size
1. Train the same model with different batch sizes (e.g., 1, 32, 128, 1024).
2. Compare:
   - Training speed
   - Validation accuracy
   - Test accuracy
   - Generalization ability
3. Which batch size leads to the **noisiest gradient updates**?
4. Which batch size generalizes better and why?

In [10]:
batch_sizes = [1, 32, 128, 1024]

batch_results = {}

for batch in batch_sizes:

    model = build_model(X_train.shape[1])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=batch,
        verbose=1
    )

    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]

    batch_results[batch] = (train_acc, val_acc, test_acc)

print(batch_results)

Epoch 1/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 102s 3ms/step - accuracy: 0.8383 - loss: 0.3400 - val_accuracy: 0.8595 - val_loss: 0.3114
Epoch 2/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 99s 3ms/step - accuracy: 0.8555 - loss: 0.3180 - val_accuracy: 0.8609 - val_loss: 0.3108
Epoch 3/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 100s 3ms/step - accuracy: 0.8592 - loss: 0.3045 - val_accuracy: 0.8619 - val_loss: 0.3098
Epoch 4/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 148s 3ms/step - accuracy: 0.8612 - loss: 0.3083 - val_accuracy: 0.8615 - val_loss: 0.3219
Epoch 5/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 100s 3ms/step - accuracy: 0.8589 - loss: 0.3100 - val_accuracy: 0.8653 - val_loss: 0.3102
Epoch 6/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 102s 3ms/step - accuracy: 0.8594 - loss: 0.3093 - val_accuracy: 0.8666 - val_loss: 0.3174
Epoch 7/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 103s 3ms/step - accuracy: 0.8636 - loss: 0.3038 - val_accuracy: 0.8616 - val_loss: 0.3137
Epoch 8/10
34189/34189 ━━━━━━━━━━━━━━━━━━━━ 101s 3ms/step - acc


## Part 3: Overfitting and Regularization
1. Train a large neural network (many parameters) on the dataset.
2. Observe training vs. validation accuracy.  
   - Do you see signs of overfitting?
3. Apply regularization techniques:
   - **L2 regularization**
   - **Dropout**
4. Compare the validation results before and after regularization.
5. Which regularization method was more effective in reducing overfitting? Why?

---


In [11]:
def build_large_model(input_dim):

    model = keras.Sequential([
        layers.Dense(256, activation="relu", input_shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    return model

Train Without Regularization

In [12]:
model = build_large_model(X_train.shape[1])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_no_reg = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8336 - loss: 0.3445 - val_accuracy: 0.8583 - val_loss: 0.3138
Epoch 2/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8550 - loss: 0.3130 - val_accuracy: 0.8631 - val_loss: 0.3066
Epoch 3/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8599 - loss: 0.3042 - val_accuracy: 0.8620 - val_loss: 0.3087
Epoch 4/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8640 - loss: 0.2938 - val_accuracy: 0.8533 - val_loss: 0.3145
Epoch 5/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8629 - loss: 0.2925 - val_accuracy: 0.8632 - val_loss: 0.3068
Epoch 6/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8705 - loss: 0.2815 - val_accuracy: 0.8606 - val_loss: 0.3133
Epoch 7/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8694 - loss: 0.2802 - val_accuracy: 0.8601 - val_loss: 0.3231
Epoch 8/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8738 - loss: 0.2687 - 

L2 Regularization

In [13]:
from tensorflow.keras.regularizers import l2

def build_l2_model(input_dim):

    model = keras.Sequential([
        layers.Dense(256, activation="relu", kernel_regularizer=l2(0.001), input_shape=(input_dim,)),
        layers.Dense(256, activation="relu", kernel_regularizer=l2(0.001)),
        layers.Dense(128, activation="relu", kernel_regularizer=l2(0.001)),
        layers.Dense(1, activation="sigmoid")
    ])

    return model

model_l2 = build_l2_model(X_train.shape[1])

model_l2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_l2 = model_l2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8316 - loss: 0.5558 - val_accuracy: 0.8542 - val_loss: 0.3468
Epoch 2/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8556 - loss: 0.3411 - val_accuracy: 0.8530 - val_loss: 0.3396
Epoch 3/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8541 - loss: 0.3328 - val_accuracy: 0.8609 - val_loss: 0.3241
Epoch 4/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8554 - loss: 0.3305 - val_accuracy: 0.8638 - val_loss: 0.3193
Epoch 5/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8553 - loss: 0.3272 - val_accuracy: 0.8624 - val_loss: 0.3199
Epoch 6/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8591 - loss: 0.3215 - val_accuracy: 0.8620 - val_loss: 0.3193
Epoch 7/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8548 - loss: 0.3221 - val_accuracy: 0.8473 - val_loss: 0.3433
Epoch 8/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8535 - loss: 0.3208 - 

Dropout Regularization

In [14]:
def build_dropout_model(input_dim):

    model = keras.Sequential([
        layers.Dense(256, activation="relu", input_shape=(input_dim,)),
        layers.Dropout(0.5),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    return model

model_dropout = build_dropout_model(X_train.shape[1])

model_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_dropout = model_dropout.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32
)

Epoch 1/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8258 - loss: 0.3726 - val_accuracy: 0.8597 - val_loss: 0.3113
Epoch 2/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8545 - loss: 0.3196 - val_accuracy: 0.8642 - val_loss: 0.3076
Epoch 3/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8531 - loss: 0.3192 - val_accuracy: 0.8612 - val_loss: 0.3059
Epoch 4/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8511 - loss: 0.3180 - val_accuracy: 0.8621 - val_loss: 0.3067
Epoch 5/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8551 - loss: 0.3118 - val_accuracy: 0.8649 - val_loss: 0.3057
Epoch 6/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8611 - loss: 0.3091 - val_accuracy: 0.8612 - val_loss: 0.3101
Epoch 7/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8586 - loss: 0.3057 - val_accuracy: 0.8628 - val_loss: 0.3046
Epoch 8/20
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8581 - loss: 0.3068 - 

## Part 4: Early Stopping
1. Train the model for many epochs without early stopping.  
   - Plot training, validation, and test curves.
2. Train again with **early stopping** (monitor validation loss).
3. Compare the number of epochs trained and the final validation/test accuracy.
4. Explain how early stopping helps prevent overfitting.

---

Without Early Stopping

In [15]:
model = build_model(X_train.shape[1])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_no_stop = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)

Epoch 1/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8286 - loss: 0.3639 - val_accuracy: 0.8493 - val_loss: 0.3199
Epoch 2/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8547 - loss: 0.3090 - val_accuracy: 0.8584 - val_loss: 0.3051
Epoch 3/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8584 - loss: 0.3031 - val_accuracy: 0.8613 - val_loss: 0.3048
Epoch 4/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8599 - loss: 0.2980 - val_accuracy: 0.8604 - val_loss: 0.3045
Epoch 5/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8590 - loss: 0.2989 - val_accuracy: 0.8587 - val_loss: 0.3065
Epoch 6/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8622 - loss: 0.2952 - val_accuracy: 0.8601 - val_loss: 0.3062
Epoch 7/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8641 - loss: 0.2922 - val_accuracy: 0.8582 - val_loss: 0.3096
Epoch 8/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8635 - loss: 0.2891 - 

With Early Stopping

In [16]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model = build_model(X_train.shape[1])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_stop = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8314 - loss: 0.3598 - val_accuracy: 0.8620 - val_loss: 0.3057
Epoch 2/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8555 - loss: 0.3092 - val_accuracy: 0.8589 - val_loss: 0.3097
Epoch 3/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8511 - loss: 0.3119 - val_accuracy: 0.8601 - val_loss: 0.3040
Epoch 4/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8548 - loss: 0.3106 - val_accuracy: 0.8619 - val_loss: 0.3042
Epoch 5/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8593 - loss: 0.3032 - val_accuracy: 0.8591 - val_loss: 0.3074
Epoch 6/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8613 - loss: 0.2973 - val_accuracy: 0.8619 - val_loss: 0.3035
Epoch 7/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8630 - loss: 0.2962 - val_accuracy: 0.8615 - val_loss: 0.3043
Epoch 8/50
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8658 - loss: 0.2915 - 

## Part 5: Reflection
1. Summarize what you learned about:
   - The role of optimizers
   - The effect of batch size
   - Regularization methods
   - Early stopping
   - Train/validation/test splits
2. If you had to train a deep learning model on a new tabular dataset, what choices would you make for:
   - Optimizer
   - Batch size
   - Regularization
   - Early stopping
   - Data splitting strategy  
   and why?

In [17]:
train_acc = model.evaluate(X_train, y_train)[1]
val_acc = model.evaluate(X_val, y_val)[1]
test_acc = model.evaluate(X_test, y_test)[1]

print("Train Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Test Accuracy:", test_acc)

1069/1069 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8619 - loss: 0.2944
229/229 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8599 - loss: 0.3099
229/229 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8623 - loss: 0.2971
Train Accuracy: 0.8619731664657593
Validation Accuracy: 0.8618618845939636
Test Accuracy: 0.8606523871421814


Which optimizer converges the fastest? Which gives the best generalization?

Adam converged the fastest during training because it adapts the learning rate for each parameter automatically Adam achieved the highest training accuracy 0.867, meaning it learned the training data quickly buttt SGD achieved the best validation and test accuracy 0.861 validation and 0.866 test, which indicates slightly better generalization. This shows that although Adam trains faster, simpler optimizers like SGD can sometimes generalize better.

Why does Adam often perform better than plain SGD?

Adam often performs better because it combines the advantages of momentum and adaptive learning rates. It keeps track of both the first and second moments of the gradients, which helps stabilize updates and adjust learning rates automatically for each parameter. This allows Adam to converge faster and handle noisy gradients better than plain SGD, especially in deep neural networks.

Which batch size leads to the noisiest gradient updates?

Batch size 1 leads  bec the model updates its weights after every single training example. This means the gradient is based on very little data, which causes high variance in the updates

Which batch size generalizes better and why?

Smaller batch sizes (such as 1 or 32) tend to generalize better because the noisy gradient updates act as a form of regularization. This noise helps the model avoid getting stuck in sharp minima and encourages it to find solutions that generalize better to unseen data. In our results, batch size 1 achieved the best validation accuracy.

Do you see signs of overfitting?

Yes, there are clear signs of overfitting. During training without regularization, the training accuracy kept increasing (up to around 0.91) while the validation accuracy started decreasing after a few epochs. At the same time, the validation loss kept increasing. This indicates that the model was memorizing the training data instead of learning patterns that generalize well.

Which regularization method was more effective in reducing overfitting? Why?

Both L2 regularization and dropout helped reduce overfitting, but L2 regularization appeared more stable in this experiment. L2 regularization penalizes large weights and forces the model to learn simpler patterns, which improves generalization. As a result, the validation accuracy stayed more consistent across epochs and the validation loss remained stable compared to the model without regularization.

Compare epochs and performance

Without early stopping, the model trained for the full 50 epochs, but the validation accuracy started to decrease after the early epochs, indicating overfitting. With early stopping, the training stopped around epoch 11, when the validation loss stopped improving. The final validation and test accuracies remained around 0.86, which shows that the model stopped training before overfitting occurred.

How does early stopping help prevent overfitting?

Early stopping prevents overfitting by stopping training when the validation performance stops improving. Instead of continuing to train and fitting noise in the training data, the model keeps the weights from the best epoch. This helps maintain better generalization to unseen data.